In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob as gb
import os
from tracking import CustomizedKalmanFilter

In [2]:
# Create a background image from a sequence of images

def create_backgorund(path, images):

    # Get 20 evenly spaced frames from the sequence of images
    background_frames = np.linspace(0, len(images) - 1, 20, dtype=int)

    # Read the selected frames and convert them to grayscale
    gray_frames = []

    for i in background_frames:
        frame = cv2.imread(images[i])
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_frames.append(gray_frame)

    # Stack the grayscale frames along a new axis and compute the median across that axis
    stacked_frames = np.stack(gray_frames, axis=0)
    median_frame = np.median(stacked_frames, axis=0).astype(dtype=np.uint8)

    return median_frame

In [3]:
# Detect husky in the image using background subtraction

def detect_husky(background, image):

    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute the absolute difference between the background and the current image
    diff = cv2.absdiff(background, image_gray)

    # Apply a binary threshold to the difference image to create a mask of the moving objects
    thresh, mask = cv2.threshold(diff, 22, 255, cv2.THRESH_BINARY)

    # Remove the top part of the mask to avoid detecting the sky or other irrelevant areas and apply morphological operations to clean up the mask
    mask[:360, :] = 0
    mask = cv2.erode(mask, None, iterations=1)
    mask = cv2.dilate(mask, None, iterations=7)
    mask = cv2.erode(mask, None, iterations=6)

    # Find contours in the mask and filter them based on area to identify the husky
    cnt = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areas = [cv2.contourArea(c) for c in cnt[0]]
    min_area = 500

    if len(areas) <= 0 or max(areas) < min_area:
        return None
    else:
        robot_cnt = cnt[0][np.argmax(areas)]

    # Compute the bounding box of the detected husky contour
    bounding_box = cv2.boundingRect(robot_cnt)

    return bounding_box

In [7]:
# Define the path to the directory containing the images
path = './3_curved_bend/3_curved_bend/left'

# Define how many frames to skip between detections (1 means every frame, 2 means every other frame, etc.)
num_frames = 1

# Get a sorted list of all PNG images in the specified directory
images = sorted(gb.glob(path + '/*.png'))

background = create_backgorund(path, images)

tracker = CustomizedKalmanFilter(1/30, False, x_init_process_variance=1, y_init_process_variance=1)

detected_images = []

# Loop through the images, detect the husky, and draw bounding boxes on the detected images
for i in range(0, len(images)):
    
    image = cv2.imread(images[i])
    output = image.copy()

    bounding_box = detect_husky(background, image)
    center_of_box = None
    w = None
    h = None
    if bounding_box is not None:
        x, y, w, h = bounding_box
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)
        center_of_box = np.array([x + (w // 2), y + (h // 2)])
        
    
    tracker.update_cycle(center_of_box)
    estimated_x = tracker.get_x()
    # print(estimated_x.ravel())
    # cv2.rectangle(output, (estimated_x[0], estimated_x[1]), (estimated_x[0] + w, estimated_x[1] + h), (255, 255, 0), 2) 
    cv2.circle(output, (int(estimated_x[0][0]), int(estimated_x[1][0])), 60, (0, 255, 255), 2)
    # Append the output image every num_frames frames
    if i % num_frames == 0:
        detected_images.append(output)

# TODO: This part need to be erased when the project is finished
# Save the detected images as a video
_, _, name, _ = path.split('/')

height, width = detected_images[0].shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

output_path = f'results/{name}_husky_detection.mp4'
os.makedirs('results', exist_ok=True)

video = cv2.VideoWriter(
    output_path,
    fourcc,
    10,
    (width, height)
)

for frame in detected_images:
    video.write(frame)

video.release()
